<a href="https://colab.research.google.com/github/Less-liee/tarea_uno/blob/main/tarea1_e2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb                                     # modelo de gradient boosting
from sklearn.model_selection import StratifiedKFold, cross_val_score   # validación cruzada

In [5]:
SEED = 42 #semilla para repetir resultados y tner un mismo resultado
gasto_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

In [6]:
train = pd.read_csv('train.csv')  #respuesta
test = pd.read_csv('test.csv')  #no mada manda, la debemos calcular
print(train.shape, test.shape)

#train+test, no da respuesta (train lo tiene, test no)
todo = pd.concat([train.drop(columns='Transported'), test], ignore_index=True)
conteo_grupo = todo['PassengerId'].str.split('_').str[0].value_counts()    # personas por grupo
conteo_apellido = todo['Name'].str.split().str[-1].value_counts()          # personas por apellido

(8693, 14) (4277, 13)


In [7]:
def preparar(df):
    df = df.copy()  #no modifica
    grupo = df['PassengerId'].str.split('_').str[0] #
    df['tam_grupo'] = grupo.map(conteo_grupo)  #cuenta
    df['tam_familia'] = df['Name'].str.split().str[-1].map(conteo_apellido).fillna(1)

    cab = df['Cabin'].str.split('/', expand=True)
    df['deck'] = cab[0]
    df['cab_num'] = pd.to_numeric(cab[1], errors='coerce')
    df['side'] = cab[2]

    for c in gasto_cols:
        df.loc[df['CryoSleep'] == True, c] = df.loc[df['CryoSleep'] == True, c].fillna(0)
    df['gasto_total'] = df[gasto_cols].sum(axis=1)
    df['sin_gasto'] = (df['gasto_total'] == 0).astype(int)
    df['log_gasto'] = np.log1p(df['gasto_total'])

    for c in ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'deck', 'side']:
        df[c] = df[c].astype(object).where(df[c].notna(), 'Desconocido').astype(str).astype('category')
    df['Age'] = df['Age'].fillna(train['Age'].median())
    return df
    #PassengerId y Cabin trean mas info

In [8]:
cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'deck', 'side', 'cab_num', 'Age',
        'tam_grupo', 'tam_familia', 'gasto_total', 'sin_gasto', 'log_gasto'] + gasto_cols

X = preparar(train)[cols]
y = train['Transported'].astype(int)  #cof
Xt = preparar(test)[cols]

#train y test comparten
for c in X.select_dtypes('category').columns:
    cats = sorted(set(X[c].cat.categories) | set(Xt[c].cat.categories))
    X[c] = pd.Categorical(X[c].astype(str), categories=cats)
    Xt[c] = pd.Categorical(Xt[c].astype(str), categories=cats)

In [9]:
modelo = lgb.LGBMClassifier(
    n_estimators=300,  #arbo
    learning_rate=0.03,  #recoore el arbol, el de abajo es mejor
    num_leaves=15,  #cpocos datos complejos
    subsample=0.8,  #usa 80%
    subsample_freq=1,  #muestra
    colsample_bytree=0.8,  #usa 80%
    min_child_samples=20,  #minimo
    random_state=SEED, verbose=-1)

cv = StratifiedKFold(5, shuffle=True, random_state=SEED)  #5 partes
acc = cross_val_score(modelo, X, y, cv=cv, scoring='accuracy')  #5 veces
print('Exactitud por fold:', acc.round(4), '| promedio:', acc.mean().round(4))
#corrige el arbol de decision, el promedio es mi estimacion lo muestra como x de cada tal

Exactitud por fold: [0.8148 0.8108 0.8166 0.8136 0.8015] | promedio: 0.8115


In [10]:
modelo.fit(X, y)  #train
pred = modelo.predict_proba(Xt)[:, 1] > 0.5  #si es mayor de 0.5 es v
sub = pd.DataFrame({'PassengerId': test['PassengerId'], 'Transported': pred})
sub.to_csv('submission_spaceship.csv', index=False)
print(sub.shape, '| ids únicos:', sub['PassengerId'].is_unique)
sub.head()
#enreno los daots

(4277, 2) | ids únicos: True


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
